# NIMBY Score Analysis: Does Local Opposition Predict Planning Delay?

**Question:** Do local authorities with high NIMBY (Not In My Back Yard) opposition scores actually see renewable energy planning applications take longer to process? We combine AI-generated NIMBY scores with the full REPD dataset and test whether measured local opposition correlates with real-world processing time — from application submission to final resolution.

**Data sources:**
- `REPD_Publication_Q4_2025.csv` — 13,900+ UK renewable energy planning records with dates
- `nimby_score.json` — 298 project analyses scoring NIMBY opposition, pettiness, organisation, and political leaning
- `localauth.json` — UK Local Authority boundary polygons

**Join key:** `nimby_score.json[refid]` ↔ `REPD[Ref ID]`

**Processing time** = days from `Planning Application Submitted` to the earliest resolution event (permission granted, refused, withdrawn, expired, or appeal outcome).

**Structure:**
1. Load & Merge — parse dates, compute processing time, join NIMBY scores to REPD
2. Project-Level Analysis — do higher NIMBY scores correlate with longer processing times?
3. Local Authority Aggregation — does average NIMBY score per LA predict its average processing time?
4. Geographic Map — choropleth comparison of average delay vs. NIMBY scores across the UK
5. Caveats — data quality, selection bias, and confounds

In [ ]:
import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import json
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import geopandas as gpd
from scipy import stats

from src.processors.repd_processor import (
    REPDProcessor,
    SUCCESSFUL_DEVELOPMENT_TYPES,
    UNSUCCESSFUL_DEVELOPMENT_TYPES,
    CANCELLED_DEVELOPMENT_TYPES,
    NEUTRAL_DEVELOPMENT_TYPES,
)

DARK_BG   = '#13100d'
DARK_AX   = '#13100d'
EDGE_COL  = '#332418'
TEXT_COL  = '#f5f0eb'
LABEL_COL = '#8a8278'
TICK_COL  = '#b8b0a6'
SUCCESS_C = '#38a870'
FAIL_C    = '#d44030'
PENDING_C = '#d4a830'
NIMBY_C   = '#9b59b6'
DELAY_C   = '#e67e22'

# Date columns representing a final planning resolution
RESOLVE_COLS = [
    'Planning Permission Granted',
    'Planning Permission Refused',
    'Planning Application Withdrawn',
    'Appeal Refused',
    'Appeal Granted',
    'Appeal Withdrawn',
    'Planning Permission Expired',
]

# National-level authorities not matchable to Local Authority Districts
NATIONAL_AUTHORITIES = {
    'The Planning Inspectorate - National Infrastructure',
    'Scottish Government (S36)',
    'Scottish Government (S37)',
    'Welsh Government',
    'Marine Management Organisation',
    'Scottish Government',
}

def style_ax(ax):
    ax.set_facecolor(DARK_AX)
    ax.tick_params(colors=TICK_COL)
    for spine in ax.spines.values():
        spine.set_edgecolor(EDGE_COL)
    ax.set_axisbelow(True)

print('Imports OK')

## 1. Load & Merge

Load the full REPD dataset, parse date columns, and compute `processing_days` — the number of days from application submission to the earliest resolution event. Then filter valid NIMBY records and inner-join on `Ref ID` = `refid`.

In [ ]:
proc   = REPDProcessor(encoding='latin-1')
df_raw = proc.load()
df_raw = proc.coordinates_to_lat_lon(df_raw)

# Parse date columns (UK format dd/mm/yyyy)
for col in ['Planning Application Submitted'] + RESOLVE_COLS:
    df_raw[col] = pd.to_datetime(df_raw[col], errors='coerce', dayfirst=True)

# Processing time = days from submission to earliest resolution event
df_raw['resolution_date'] = df_raw[RESOLVE_COLS].min(axis=1)
df_raw['processing_days'] = (
    df_raw['resolution_date'] - df_raw['Planning Application Submitted']
).dt.days

valid_days = df_raw.loc[df_raw['processing_days'] > 0, 'processing_days']
print(f'REPD records loaded:              {len(df_raw):,}')
print(f'Records with valid processing time: {len(valid_days):,}')
print(f'\nProcessing time distribution:')
print(f'  Median:  {valid_days.median():.0f} days  ({valid_days.median()/365:.1f} yrs)')
print(f'  Mean:    {valid_days.mean():.0f} days  ({valid_days.mean()/365:.1f} yrs)')
print(f'  P25–P75: {valid_days.quantile(0.25):.0f} – {valid_days.quantile(0.75):.0f} days')
print(f'  Max:     {valid_days.max():.0f} days  ({valid_days.max()/365:.1f} yrs)')

with open('src/data/nimby_score.json', 'r') as f:
    nimby_data = json.load(f)
df_nimby = pd.DataFrame(nimby_data)

print(f'\nNIMBY raw records:               {len(df_nimby):,}')
print(f'Zero-score (PROJECT NOT FOUND):  {(df_nimby["Nimby Score"] == 0).sum():,}')
print(f'Valid scored records:             {(df_nimby["Nimby Score"] > 0).sum():,}')

In [ ]:
# Filter to valid NIMBY analyses
df_nimby_valid = df_nimby[
    (df_nimby['Nimby Score'] > 0) & (df_nimby['Accuracy Score'] > 0)
].copy()
df_nimby_valid['refid'] = pd.to_numeric(df_nimby_valid['refid'], errors='coerce').astype('Int64')

df_raw['Ref ID'] = pd.to_numeric(df_raw['Ref ID'], errors='coerce').astype('Int64')

# Inner join — processing_days comes through automatically from df_raw
df_merged = df_raw.merge(
    df_nimby_valid[['refid', 'Nimby Score', 'Accuracy Score',
                    'Petty Score', 'Organized Score', 'Political Leaning']],
    left_on='Ref ID', right_on='refid',
    how='inner'
)

# Outcome column kept for scatter colouring
df_merged['cancelled'] = df_merged['Development Status (short)'].isin(CANCELLED_DEVELOPMENT_TYPES).astype(int)

df_scored = df_merged[df_merged['processing_days'] > 0].copy()

print(f'Merged records (all):           {len(df_merged):,}')
print(f'Merged with valid process time: {len(df_scored):,}')
print(f'\nDevelopment Status breakdown (all merged):')
print(df_merged['Development Status (short)'].value_counts().to_string())
print(f'\nMedian processing time in scored set: {df_scored["processing_days"].median():.0f} days')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=DARK_BG)

# Left: NIMBY score distribution
ax1 = axes[0]
style_ax(ax1)
ax1.hist(df_nimby_valid['Nimby Score'], bins=20, color=NIMBY_C, alpha=0.85,
         edgecolor=DARK_BG, linewidth=0.5, zorder=3)
med_nimby = df_nimby_valid['Nimby Score'].median()
ax1.axvline(med_nimby, color=PENDING_C, linewidth=1.8, linestyle='--',
            label=f'Median: {med_nimby:.0f}', zorder=4)
ax1.set_xlabel('NIMBY Score', color=LABEL_COL, fontsize=11)
ax1.set_ylabel('Count', color=LABEL_COL, fontsize=11)
ax1.set_title('NIMBY Score Distribution\n(valid scored projects)', color=TEXT_COL, fontsize=12, fontweight='bold')
ax1.grid(axis='y', color='#ffffff', alpha=0.07)
ax1.legend(fontsize=10, framealpha=0.3, facecolor='#1d1810', edgecolor=EDGE_COL, labelcolor=TICK_COL)

# Right: processing time distribution, split by outcome
ax2 = axes[1]
style_ax(ax2)
bins = np.linspace(0, df_scored['processing_days'].quantile(0.97), 30)
for outcome, label, colour in [
    (0, 'Not Cancelled', SUCCESS_C),
    (1, 'Cancelled',     FAIL_C),
]:
    grp = df_scored[df_scored['cancelled'] == outcome]['processing_days'] / 365
    if len(grp) > 0:
        ax2.hist(grp, bins=bins / 365, color=colour, alpha=0.65,
                 edgecolor=DARK_BG, linewidth=0.3, label=f'{label}  (n={len(grp)})', zorder=3)
med_days = df_scored['processing_days'].median() / 365
ax2.axvline(med_days, color=PENDING_C, linewidth=1.8, linestyle='--',
            label=f'Overall median: {med_days:.1f} yrs', zorder=4)
ax2.set_xlabel('Processing Time (years)', color=LABEL_COL, fontsize=11)
ax2.set_ylabel('Count', color=LABEL_COL, fontsize=11)
ax2.set_title('Planning Processing Time\n(scored projects with resolved outcomes)', color=TEXT_COL, fontsize=12, fontweight='bold')
ax2.grid(axis='y', color='#ffffff', alpha=0.07)
ax2.legend(fontsize=9, framealpha=0.3, facecolor='#1d1810', edgecolor=EDGE_COL, labelcolor=TICK_COL)

fig.text(0.99, 0.01, 'Damian Bemben - bemben.co.uk', ha='right', va='bottom',
         fontsize=7, color='#54483e', alpha=0.7)
plt.tight_layout()
plt.show()

## 2. Project-Level Analysis

Does a higher NIMBY score at the project level correspond to a longer planning process? We look at this two ways:
1. **Quartile box plots** — bin projects by NIMBY score and compare processing times across the four quartiles (Mann-Whitney Q1 vs Q4).
2. **Regression scatter** — NIMBY Score vs processing time (years) with a linear fit and Spearman rank correlation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=DARK_BG)

# Left: processing time by NIMBY score quartile
ax1 = axes[0]
style_ax(ax1)

try:
    df_scored['nimby_q'] = pd.qcut(
        df_scored['Nimby Score'], q=4,
        labels=['Q1\n(Low NIMBY)', 'Q2', 'Q3', 'Q4\n(High NIMBY)']
    )
    q_labels = ['Q1\n(Low NIMBY)', 'Q2', 'Q3', 'Q4\n(High NIMBY)']
    q_data   = [df_scored[df_scored['nimby_q'] == q]['processing_days'].values / 365
                for q in q_labels]
    q_counts = [len(d) for d in q_data]

    purple_shades = ['#5b2c6f', '#7d3c98', '#9b59b6', '#c39bd3']
    bp = ax1.boxplot(q_data, patch_artist=True, widths=0.55,
                     medianprops=dict(color=PENDING_C, linewidth=2.5),
                     whiskerprops=dict(color=TICK_COL, linewidth=0.9),
                     capprops=dict(color=TICK_COL),
                     flierprops=dict(marker='o', markersize=3, alpha=0.4,
                                    markerfacecolor=LABEL_COL, markeredgecolor='none'),
                     boxprops=dict(edgecolor=EDGE_COL))
    for patch, colour in zip(bp['boxes'], purple_shades):
        patch.set_facecolor(colour + 'aa')

    ax1.set_xticks(range(1, 5))
    ax1.set_xticklabels(
        [f'{lbl}\nn={n}' for lbl, n in zip(q_labels, q_counts)],
        fontsize=8, color=TICK_COL
    )

    if q_counts[0] > 1 and q_counts[3] > 1:
        _, p = stats.mannwhitneyu(q_data[0], q_data[3], alternative='two-sided')
        sig_str = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        p_col = PENDING_C if p < 0.05 else LABEL_COL
        ax1.text(0.5, 0.97, f'Q1 vs Q4: p = {p:.3f}  {sig_str}',
                 transform=ax1.transAxes, ha='center', va='top', fontsize=8, color=p_col,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='#1d1810', edgecolor=EDGE_COL, alpha=0.7))
except ValueError:
    ax1.text(0.5, 0.5, 'Insufficient data\nfor quartile split',
             transform=ax1.transAxes, ha='center', va='center', fontsize=10, color=LABEL_COL)

ax1.set_ylabel('Processing Time (years)', color=LABEL_COL, fontsize=11)
ax1.set_title('Processing Time by NIMBY Score Quartile', color=TEXT_COL, fontsize=12, fontweight='bold')
ax1.grid(axis='y', color='#ffffff', alpha=0.07)

# Right: processing time by outcome group
ax2 = axes[1]
style_ax(ax2)

outcome_groups = [
    (df_scored[df_scored['Development Status (short)'].isin(SUCCESSFUL_DEVELOPMENT_TYPES)]['processing_days'].values / 365,
     'Successful', SUCCESS_C),
    (df_scored[df_scored['cancelled'] == 1]['processing_days'].values / 365,
     'Cancelled', FAIL_C),
]
group_data   = [g[0] for g in outcome_groups if len(g[0]) > 0]
group_labels = [g[1] for g in outcome_groups if len(g[0]) > 0]
group_cols   = [g[2] for g in outcome_groups if len(g[0]) > 0]

if group_data:
    bp2 = ax2.boxplot(group_data, patch_artist=True, widths=0.5,
                      medianprops=dict(color=PENDING_C, linewidth=2.5),
                      whiskerprops=dict(color=TICK_COL, linewidth=0.9),
                      capprops=dict(color=TICK_COL),
                      flierprops=dict(marker='o', markersize=3, alpha=0.4,
                                     markerfacecolor=LABEL_COL, markeredgecolor='none'),
                      boxprops=dict(edgecolor=EDGE_COL))
    for patch, colour in zip(bp2['boxes'], group_cols):
        patch.set_facecolor(colour + '55')
    ax2.set_xticks(range(1, len(group_labels) + 1))
    ax2.set_xticklabels(
        [f'{lbl}\nn={len(d)}' for lbl, d in zip(group_labels, group_data)],
        fontsize=9, color=TICK_COL
    )
    if len(group_data) >= 2 and len(group_data[0]) > 1 and len(group_data[1]) > 1:
        _, p2 = stats.mannwhitneyu(group_data[0], group_data[1], alternative='two-sided')
        sig2 = '***' if p2 < 0.001 else ('**' if p2 < 0.01 else ('*' if p2 < 0.05 else 'ns'))
        p_col2 = PENDING_C if p2 < 0.05 else LABEL_COL
        ax2.text(0.5, 0.97, f'p = {p2:.3f}  {sig2}', transform=ax2.transAxes,
                 ha='center', va='top', fontsize=8, color=p_col2,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='#1d1810', edgecolor=EDGE_COL, alpha=0.7))

ax2.set_ylabel('Processing Time (years)', color=LABEL_COL, fontsize=11)
ax2.set_title('Processing Time by Planning Outcome\n(scored projects)', color=TEXT_COL, fontsize=12, fontweight='bold')
ax2.grid(axis='y', color='#ffffff', alpha=0.07)

fig.suptitle('Does NIMBY Score Predict Longer Planning Processes?',
             color=TEXT_COL, fontsize=14, fontweight='bold')
fig.text(0.99, 0.01, 'Damian Bemben - bemben.co.uk', ha='right', va='bottom',
         fontsize=7, color='#54483e', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7), facecolor=DARK_BG)
style_ax(ax)

for outcome, label, colour in [(0, 'Not Cancelled', SUCCESS_C), (1, 'Cancelled', FAIL_C)]:
    grp = df_scored[df_scored['cancelled'] == outcome]
    ax.scatter(
        grp['Nimby Score'], grp['processing_days'] / 365,
        c=colour, label=f'{label}  (n={len(grp)})',
        alpha=0.70, s=65, edgecolors='#ffffff', linewidths=0.3, zorder=3
    )

# Regression line across all scored projects with valid processing time
x_all = df_scored['Nimby Score'].values
y_all = df_scored['processing_days'].values / 365
if len(x_all) >= 3:
    slope, intercept, r_val, p_val, _ = stats.linregress(x_all, y_all)
    x_line = np.linspace(x_all.min(), x_all.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, color=DELAY_C,
            linewidth=2, linestyle='--', alpha=0.8, zorder=2,
            label=f'Linear fit  (r = {r_val:.3f}, p = {p_val:.3f})')
    sp_r, sp_p = stats.spearmanr(x_all, y_all)
    ax.text(0.98, 0.05,
            f'Spearman \u03c1 = {sp_r:+.3f},  p = {sp_p:.3f}',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=9, color=PENDING_C if sp_p < 0.05 else LABEL_COL)

ax.set_xlabel('NIMBY Score', color=LABEL_COL, fontsize=12)
ax.set_ylabel('Processing Time (years)', color=LABEL_COL, fontsize=12)
ax.set_title('NIMBY Score vs. Planning Processing Time\nColoured by Outcome',
             color=TEXT_COL, fontsize=13, fontweight='bold')
ax.grid(color='#ffffff', alpha=0.07)
ax.legend(fontsize=9, framealpha=0.3, facecolor='#1d1810', edgecolor=EDGE_COL, labelcolor=TICK_COL)

fig.text(0.99, 0.01, 'Damian Bemben - bemben.co.uk', ha='right', va='bottom',
         fontsize=7, color='#54483e', alpha=0.7)
plt.tight_layout()
plt.show()

## 3. Local Authority Aggregation

Aggregate the full REPD dataset by Planning Authority to compute average processing time, then join with average NIMBY scores from the scored subset. Test whether high-NIMBY LAs have systematically longer planning processes.

In [ ]:
# Average processing time per LA — all REPD projects with valid dates
df_la = df_raw[
    ~df_raw['Planning Authority'].isin(NATIONAL_AUTHORITIES) &
    (df_raw['processing_days'] > 0)
].copy()

la_stats = (
    df_la
    .groupby('Planning Authority')
    .agg(
        avg_days   =('processing_days', 'mean'),
        median_days=('processing_days', 'median'),
        project_count=('processing_days', 'count'),
    )
    .query('project_count >= 3')
    .sort_values('avg_days', ascending=False)
)
la_stats['avg_years']    = la_stats['avg_days']    / 365
la_stats['median_years'] = la_stats['median_days'] / 365

print(f'LAs with \u22653 resolved projects: {len(la_stats):,}')
print(f'\nTop 10 — longest average processing time:')
print(la_stats.head(10)[['project_count', 'avg_years', 'median_years']].to_string())
print(f'\nBottom 10 — shortest average processing time:')
print(la_stats.tail(10)[['project_count', 'avg_years', 'median_years']].to_string())

In [ ]:
# Average NIMBY scores per LA (from the merged NIMBY-scored set)
la_nimby = (
    df_merged.groupby('Planning Authority')
    .agg(
        avg_nimby   =('Nimby Score', 'mean'),
        avg_petty   =('Petty Score', 'mean'),
        avg_organized=('Organized Score', 'mean'),
        nimby_count =('Nimby Score', 'count'),
    )
)

la_combined = la_stats.join(la_nimby, how='inner')
print(f'LAs with both processing time data and NIMBY scores: {len(la_combined)}')
if len(la_combined) > 0:
    print(la_combined[['project_count', 'avg_years', 'avg_nimby', 'nimby_count']]
          .sort_values('avg_nimby', ascending=False).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=DARK_BG)

# Left: scatter — avg NIMBY score vs avg processing time
ax1 = axes[0]
style_ax(ax1)

if len(la_combined) > 0:
    sizes = np.clip(la_combined['project_count'] * 6, 30, 400)
    sc = ax1.scatter(
        la_combined['avg_nimby'], la_combined['avg_years'],
        s=sizes, c=la_combined['avg_nimby'], cmap='RdPu', vmin=0, vmax=100,
        alpha=0.85, edgecolors='#ffffff', linewidths=0.4, zorder=3
    )
    if len(la_combined) >= 3:
        slope, intercept, r_val, p_val, _ = stats.linregress(
            la_combined['avg_nimby'], la_combined['avg_years']
        )
        x_line = np.linspace(la_combined['avg_nimby'].min(), la_combined['avg_nimby'].max(), 100)
        p_col = PENDING_C if p_val < 0.05 else LABEL_COL
        ax1.plot(x_line, slope * x_line + intercept, color=PENDING_C,
                 linewidth=1.5, linestyle='--', alpha=0.7, zorder=2,
                 label=f'r = {r_val:.3f},  p = {p_val:.3f}')
        ax1.legend(fontsize=9, framealpha=0.3, facecolor='#1d1810',
                   edgecolor=EDGE_COL, labelcolor=p_col)
    for la_name, row in la_combined.nlargest(min(5, len(la_combined)), 'avg_nimby').iterrows():
        short = la_name[:22] + '\u2026' if len(la_name) > 22 else la_name
        ax1.annotate(short, (row['avg_nimby'], row['avg_years']),
                     fontsize=6, color=TICK_COL, xytext=(5, 0), textcoords='offset points')
    cbar = fig.colorbar(sc, ax=ax1, fraction=0.04, pad=0.02)
    cbar.set_label('Avg NIMBY Score', color=TICK_COL, fontsize=8)
    cbar.ax.yaxis.set_tick_params(color=TICK_COL)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TICK_COL)
    cbar.outline.set_edgecolor(EDGE_COL)

ax1.text(0.02, 0.97, f'n = {len(la_combined)} local authorities\n(bubble size \u221d project count)',
         transform=ax1.transAxes, ha='left', va='top', fontsize=7, color=LABEL_COL)
ax1.set_xlabel('Average NIMBY Score  (per local authority)', color=LABEL_COL, fontsize=11)
ax1.set_ylabel('Average Processing Time (years)', color=LABEL_COL, fontsize=11)
ax1.set_title('Average NIMBY Score vs.\nAverage Planning Processing Time by Local Authority',
              color=TEXT_COL, fontsize=12, fontweight='bold')
ax1.grid(color='#ffffff', alpha=0.07)

# Right: top LAs by avg NIMBY with processing time overlay
ax2 = axes[1]
style_ax(ax2)

if len(la_combined) > 0:
    n_show     = min(12, len(la_combined))
    top_la     = la_combined.nlargest(n_show, 'avg_nimby')
    x          = np.arange(len(top_la))
    ax2.bar(x, top_la['avg_nimby'], color=NIMBY_C, alpha=0.8, zorder=3, label='Avg NIMBY Score')
    short_names = [n[:18] + '\u2026' if len(n) > 18 else n for n in top_la.index]
    ax2.set_xticks(x)
    ax2.set_xticklabels(short_names, rotation=38, ha='right', fontsize=7, color=TICK_COL)
    ax2.set_ylabel('Average NIMBY Score', color=LABEL_COL, fontsize=10)
    ax2.grid(axis='y', color='#ffffff', alpha=0.07, zorder=0)

    ax2b = ax2.twinx()
    ax2b.set_facecolor(DARK_AX)
    ax2b.plot(x, top_la['avg_years'], color=DELAY_C, linewidth=2,
              marker='o', markersize=5, zorder=4, label='Avg Processing Time (yrs)')
    ax2b.set_ylabel('Avg Processing Time (years)', color=DELAY_C, fontsize=10)
    ax2b.tick_params(axis='y', colors=DELAY_C)
    ax2b.spines['right'].set_edgecolor(DELAY_C)
    for sp in ['top', 'left', 'bottom']:
        ax2b.spines[sp].set_edgecolor(EDGE_COL)

    lines1, lbl1 = ax2.get_legend_handles_labels()
    lines2, lbl2 = ax2b.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, lbl1 + lbl2, fontsize=8, framealpha=0.3,
               facecolor='#1d1810', edgecolor=EDGE_COL, labelcolor=TICK_COL)

ax2.set_title(f'Top {min(12, len(la_combined))} Local Authorities by NIMBY Score\nvs. Average Processing Time',
              color=TEXT_COL, fontsize=12, fontweight='bold')

fig.text(0.99, 0.01, 'Damian Bemben - bemben.co.uk', ha='right', va='bottom',
         fontsize=7, color='#54483e', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
if len(la_combined) >= 3:
    pearson_r,  pearson_p  = stats.pearsonr( la_combined['avg_nimby'], la_combined['avg_years'])
    spearman_r, spearman_p = stats.spearmanr(la_combined['avg_nimby'], la_combined['avg_years'])

    print('=' * 62)
    print('Local Authority Correlation: NIMBY Score vs. Processing Time')
    print('=' * 62)
    print(f'  Sample: {len(la_combined)} LAs (\u22653 resolved projects + \u22651 scored project)')
    print()
    print(f'  Pearson  r = {pearson_r:+.4f}   p = {pearson_p:.4f}  {"*" if pearson_p < 0.05 else "(ns)"}')
    print(f'  Spearman r = {spearman_r:+.4f}   p = {spearman_p:.4f}  {"*" if spearman_p < 0.05 else "(ns)"}')
    print()
    if pearson_p < 0.05:
        r2 = pearson_r ** 2
        print(f'  Significant Pearson correlation found.')
        print(f'  R\u00b2 = {r2:.4f}  \u2192  NIMBY score explains {r2*100:.1f}% of variance in avg processing time')
    else:
        print('  No statistically significant linear correlation at \u03b1 = 0.05')
        print('  (Likely reflects sparse NIMBY coverage rather than absence of effect)')
else:
    print(f'Too few overlapping LAs ({len(la_combined)}) for robust correlation analysis.')
    print('Expanding the NIMBY scoring dataset would increase statistical power.')

## 4. Geographic Map

Two side-by-side choropleths: average planning processing time across all LAs with ≥3 resolved projects (left), and average NIMBY score for LAs where projects were scored (right). Comparing the two maps shows whether the geographies of delay and opposition align.

In [ ]:
lauth     = gpd.read_file('src/data/localauth.json')
lad_names = lauth['LAD24NM'].tolist()

def fuzzy_la_match(name, lad_names, cutoff=0.55):
    matches = difflib.get_close_matches(str(name), lad_names, n=1, cutoff=cutoff)
    return matches[0] if matches else None

# Spatial join for processing time
la_stats_map = la_stats.reset_index().copy()
la_stats_map['LAD24NM'] = la_stats_map['Planning Authority'].apply(
    lambda x: fuzzy_la_match(x, lad_names)
)
matched_delay = la_stats_map.dropna(subset=['LAD24NM'])
print(f'Processing time data: {len(matched_delay)} / {len(la_stats_map)} LAs matched to boundaries')

# Spatial join for NIMBY scores
la_nimby_map = la_nimby.reset_index().copy()
la_nimby_map['LAD24NM'] = la_nimby_map['Planning Authority'].apply(
    lambda x: fuzzy_la_match(x, lad_names)
)
matched_nimby = la_nimby_map.dropna(subset=['LAD24NM'])
print(f'NIMBY score data:     {len(matched_nimby)} / {len(la_nimby_map)} LAs matched to boundaries')

gdf_delay = lauth.merge(matched_delay[['LAD24NM', 'avg_years']], on='LAD24NM', how='left')
gdf_nimby = lauth.merge(matched_nimby[['LAD24NM', 'avg_nimby']], on='LAD24NM', how='left')

# Colour scale for delay: 0 to 95th percentile
vmax_years = float(matched_delay['avg_years'].quantile(0.95)) if len(matched_delay) > 0 else 5

fig, axes = plt.subplots(1, 2, figsize=(20, 15), facecolor=DARK_BG)
for ax in axes:
    ax.set_facecolor(DARK_BG)
    ax.set_xlim(-8.5, 3.0)
    ax.set_ylim(49.8, 61.0)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for sp in ax.spines.values():
        sp.set_edgecolor(EDGE_COL)

# Left: average processing time choropleth
ax1 = axes[0]
lauth.plot(ax=ax1, facecolor='#1d1810', edgecolor='#4a3828', linewidth=0.3, zorder=1)
has_delay = gdf_delay.dropna(subset=['avg_years'])
if len(has_delay) > 0:
    has_delay.plot(ax=ax1, column='avg_years', cmap='YlOrRd',
                   vmin=0, vmax=vmax_years, alpha=0.85, zorder=2, legend=False)
sm1 = plt.cm.ScalarMappable(cmap='YlOrRd', norm=mcolors.Normalize(0, vmax_years))
sm1.set_array([])
cb1 = fig.colorbar(sm1, ax=ax1, fraction=0.025, pad=0.02)
cb1.set_label('Avg Processing Time (years)', color=TICK_COL, fontsize=9)
cb1.ax.yaxis.set_tick_params(color=TICK_COL)
plt.setp(cb1.ax.yaxis.get_ticklabels(), color=TICK_COL)
cb1.outline.set_edgecolor(EDGE_COL)
ax1.set_title('Average Planning Processing Time\nby Local Authority  (\u22653 resolved projects)',
              color=TEXT_COL, fontsize=13, fontweight='bold')

# Right: average NIMBY score choropleth
ax2 = axes[1]
lauth.plot(ax=ax2, facecolor='#1d1810', edgecolor='#4a3828', linewidth=0.3, zorder=1)
has_nimby = gdf_nimby.dropna(subset=['avg_nimby'])
if len(has_nimby) > 0:
    has_nimby.plot(ax=ax2, column='avg_nimby', cmap='RdPu',
                   vmin=0, vmax=100, alpha=0.85, zorder=2, legend=False)
sm2 = plt.cm.ScalarMappable(cmap='RdPu', norm=mcolors.Normalize(0, 100))
sm2.set_array([])
cb2 = fig.colorbar(sm2, ax=ax2, fraction=0.025, pad=0.02)
cb2.set_label('Average NIMBY Score', color=TICK_COL, fontsize=9)
cb2.ax.yaxis.set_tick_params(color=TICK_COL)
plt.setp(cb2.ax.yaxis.get_ticklabels(), color=TICK_COL)
cb2.outline.set_edgecolor(EDGE_COL)
ax2.set_title('Average NIMBY Score\nby Local Authority  (scored projects only)',
              color=TEXT_COL, fontsize=13, fontweight='bold')

fig.suptitle('Planning Delay & NIMBY Scores Across the UK',
             color=TEXT_COL, fontsize=15, fontweight='bold', y=0.995)
fig.text(0.5, 0.005,
         'Grey = no data.  Left: all LAs with \u22653 resolved projects.  Right: LAs with at least one NIMBY-scored project.',
         ha='center', va='bottom', fontsize=8, color=LABEL_COL)
fig.text(0.99, 0.005, 'Damian Bemben - bemben.co.uk', ha='right', va='bottom',
         fontsize=7, color='#54483e', alpha=0.7)
plt.tight_layout()
plt.show()

## 5. Caveats

**Selection bias.** The 298 scored projects were selected because they generated enough press coverage to be analysed — the *controversial* cases. Projects that sailed through planning rarely make the news, so scored projects will skew toward longer processing times regardless of their NIMBY score.

**Sparse NIMBY coverage.** Only ~2% of REPD projects have NIMBY scores. Most LAs contribute just 1–3 scored projects, making LA-level averages noisy. The scatter in Section 3 is indicative rather than conclusive.

**Processing time definition.** We use the *earliest* resolution date across all resolution columns. This can undercount delay if, for example, a permission was granted, expired, and re-applied for. It also excludes the construction/commissioning phase.

**Technology confound.** Onshore wind takes longer and faces more organised opposition than rooftop solar. LAs with more wind applications will appear both more "NIMBY" and slower, inflating any apparent correlation between the two metrics.

**Directionality.** Projects that took a long time were also more likely to attract press coverage and hence a NIMBY score. Slow projects and high NIMBY scores may both be symptoms of controversy, not cause and effect.

**Next steps.** Controlling for technology type (run the correlation within technology buckets) would separate the NIMBY signal from the technology-driven delay signal.